# Attention Lab 4 · Masks, Causal & Sparse Patterns

> **Types of Attention — Lab 4.** Run top to bottom (Runtime → Run all). Read the plain-English notes, run the code,
> then do the **Your turn 🧪** cell. Everything is built from scratch with NumPy so nothing is hidden.

### In plain English
Letting every word look at every other word is thorough but expensive — for a long document it's like
making everyone at a huge conference meet everyone else. So we restrict **who may look at whom** with a *mask*.

- **Causal:** a word may only look **backward**. Required to write text one word at a time (you can't use a word you
  haven't written yet — that's phone autocomplete).
- **Sliding window:** each word only checks its **nearby neighbors**. Cheap, and information still spreads across layers
  like gossip crossing a room.
- **Dilated / block / global+local:** skip around, group into chunks, or keep a few "announcement" words that everyone
  hears. These read book-length text at a fraction of the cost.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)
N = 16

## 1 · Build the masks (True = allowed to look)

In [ ]:
i, j = np.indices((N, N))

full     = np.ones((N,N), bool)
causal   = j <= i                                   # only the past
window   = np.abs(i-j) <= 2                         # local band
dilated  = (np.abs(i-j) % 2 == 0) & (np.abs(i-j) <= 6)
block    = (i // 4) == (j // 4)                     # fixed chunks
glob     = window.copy(); glob[:2,:] = True; glob[:,:2] = True   # 2 "announcement" tokens

masks = {"full":full, "causal":causal, "sliding window":window,
         "dilated":dilated, "block":block, "global+local":glob}

for name, m in masks.items():
    print(f"{name:15s} density {m.mean()*100:5.1f}%  ->  relative cost {m.mean():.2f}x")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10,6.6))
for ax, (name, m) in zip(axes.ravel(), masks.items()):
    ax.imshow(m, cmap="BuPu"); ax.set_title(f"{name}  ({m.mean()*100:.0f}%)")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("who may attend to whom  (row = looking, column = looked at)", y=1.01)
plt.tight_layout(); plt.show()

## 2 · Applying a mask: blocked spots get −∞ before softmax
That makes their probability exactly zero, so no attention leaks through.

In [ ]:
rng = np.random.default_rng(0)
d = 8
X = rng.normal(size=(N, d))

def attention(X, mask=None):
    s = X @ X.T / np.sqrt(d)
    if mask is not None:
        s = np.where(mask, s, -np.inf)     # blocked -> -infinity
    return softmax(s)

W_full   = attention(X)
W_causal = attention(X, causal)
print("row 3, full attention :", np.round(W_full[3], 2))
print("row 3, causal         :", np.round(W_causal[3], 2), " <- zeros after position 3")
assert np.allclose(W_causal[3, 4:], 0), "causal mask leaked!"
print("\n✅ causal mask verified: no attention to the future")

## 3 · Why causal masking = generation
A language model predicts word *t+1* from words *1..t*. If it could peek at word *t+1* while training, it would just
copy the answer and learn nothing.

In [ ]:
for t in range(4):
    allowed = [k for k in range(N) if causal[t,k]]
    print(f"when predicting word {t+1}, the model may look at positions {allowed}")

## 4 · The cost saving, at scale

In [ ]:
for n in [512, 2048, 8192]:
    full_cells   = n*n
    window_cells = n*5          # window of +/-2
    print(f"seq len {n:>5}: full = {full_cells:>12,} cells | window = {window_cells:>9,} cells"
          f" | {full_cells/window_cells:>7.0f}x cheaper")

## Your turn 🧪
1. Build a **causal + sliding-window** mask (both conditions). What's its density? (This is what Mistral uses.)
2. With the `global+local` mask, which rows are fully dense? Why are those tokens special?
3. Prove information still spreads: with a window of ±1, how many *layers* until word 0 can influence word 15?

In [ ]:
# Your turn: causal + local, and how far info spreads per layer
causal_local = causal & window
print("causal+local density:", f"{causal_local.mean()*100:.1f}%")
w = 1; layers = int(np.ceil(15 / w))
print(f"with window ±{w}, word 0 reaches word 15 after {layers} layers")